# Steel Plate Fault Classification

## The problem

Steel plates are flat pieces of steel used to make products and structures, from appliances and vehicles to machinery and buildings. A **fault** is an unwanted imperfection on a plate's surface, such as a scratch, stain, bump, or contamination. Manufacturers inspect plates for faults because they can affect a product's appearance, quality, reliability, or safety. Detecting faults automatically also helps people inspect many plates more quickly and consistently.

Knowing that a plate has a fault is useful; knowing **which kind** is even more useful. Different fault types can have different likely causes, such as a problem in a particular manufacturing step, a material issue, or contamination. The right response might be to repair the plate, sort it for a different use, or investigate the production process. Your job is to predict the kind of fault from measurements extracted from an image of the plate.

## Features

Each row represents one detected fault. The 24 model features are measurements extracted from its image. In the descriptions below, `X` and `Y` are pixel-coordinate directions in the image, and *luminosity* means brightness. The original dataset names several precomputed `*_Index` measures without publishing their exact formulas, so their descriptions explain the quantity they summarize rather than a calculation you need to reproduce.

| Column | Meaning |
| --- | --- |
| `id` | A row identifier used to match your prediction to the correct test row in Kaggle. Do **not** use it as a model feature. |
| `X_Minimum`, `X_Maximum` | The smallest and largest horizontal pixel coordinates of the detected fault. |
| `Y_Minimum`, `Y_Maximum` | The smallest and largest vertical pixel coordinates of the detected fault. |
| `Pixels_Areas` | The number of image pixels belonging to the fault; a measure of its area. |
| `X_Perimeter`, `Y_Perimeter` | Measurements of the fault boundary in the horizontal and vertical directions. |
| `Sum_of_Luminosity` | The total brightness of pixels in the detected fault. |
| `Minimum_of_Luminosity`, `Maximum_of_Luminosity` | The darkest and brightest pixel values in the fault. |
| `Length_of_Conveyer` | The length of the conveyor associated with the inspection measurement. |
| `TypeOfSteel_A300`, `TypeOfSteel_A400` | Binary indicators for the plate's steel type: 1 means that type applies and 0 means it does not. |
| `Steel_Plate_Thickness` | The thickness of the steel plate. |
| `Edges_Index` | A precomputed image measure describing the fault's boundary or edges. |
| `Empty_Index` | A precomputed image measure related to empty space around or within the fault region. |
| `Square_Index` | A precomputed shape measure of how square-like the fault is. |
| `Outside_X_Index` | A precomputed horizontal measure of the fault's outside region or extent. |
| `Edges_X_Index`, `Edges_Y_Index` | Precomputed edge measures in the horizontal and vertical directions. |
| `Outside_Global_Index` | A precomputed overall measure of the fault's outside region. |
| `Orientation_Index` | A precomputed measure of the fault's direction or orientation. |
| `Luminosity_Index` | A precomputed brightness-related measure. |
| `SigmoidOfAreas` | A sigmoid (S-shaped) transformation of the fault area, which compresses large area values. |

## Labels

In `train.csv`, the `fault_type` column is the label you are trying to predict. The possible labels are:

- `Pastry`: an irregular surface pattern, often described as swirls, loops, or a pattern resembling a baked pastry.
- `Z_Scratch`: a scuff or abrasion with a Z-shaped pattern.
- `K_Scatch`: a scoring mark or scratch with a K-shaped pattern. The spelling `K_Scatch` is part of the original dataset, so use it exactly.
- `Stains`: discolored spots or marks caused by impurities or chemical reactions during handling or manufacturing.
- `Dirtiness`: foreign material on the surface, such as dust, oil residue, or other debris.
- `Bumps`: small raised patches or lumps that protrude from the surface.
- `Other_Faults`: a catch-all category for faults not covered by the other six labels.

These descriptions come from a [published engineering case study using this dataset](https://rpsonline.com.sg/proceedings/esrel-sra-e2025/pdf/ESREL-SRA-E2025-P7259.pdf). For this competition, treat the seven labels as distinct categories to learn from the measurements.

## Competition restrictions

Your final classifier must be **only one** of these three model families: **k-nearest neighbors (kNN), random forest, or logistic regression**. You may tune their hyperparameters, but do not use another classifier or blend predictions from multiple classifiers.

You may do as much feature engineering as you wish. For example, you can create widths and heights from the minimum and maximum coordinates, calculate ratios or interactions, transform skewed features, select features, and standardize features when appropriate.

## How Kaggle scoring works

You receive labeled training data and an unlabeled test set. You train your model on `train.csv`, predict `fault_type` for every row in `test.csv`, and upload a CSV with the `id` and `fault_type` columns. Kaggle compares your predictions with test labels that you cannot see.

Before the competition ends, the leaderboard displays your accuracy on **half of the hidden test set**. Use it as a progress check, but avoid repeatedly changing your model just to chase a leaderboard score. Keep a validation set from the training data to make decisions about your model.

After the competition ends, we will reveal final scores using the **entire hidden test set**.

This notebook makes a simple baseline prediction using a 1-nearest-neighbor classifier and saves it in the Kaggle submission format.

In [ ]:
from pathlib import Path
import sys

import pandas as pd
from sklearn.neighbors import KNeighborsClassifier

# When this notebook is opened in Google Colab, download the competition data.
path = 'projects/proj02'
if 'google.colab' in sys.modules:
    !wget -q -O /content/course.zip https://github.com/dsc-courses/cosmos-ml-cluster-2026/archive/refs/heads/main.zip
    !unzip -q -o /content/course.zip 'cosmos-ml-cluster-2026-main/{path}/data/*' -d /content/course-assets
    !cp -R /content/course-assets/cosmos-ml-cluster-2026-main/{path}/data .

In [ ]:
train = pd.read_csv('data/train.csv')
test = pd.read_csv('data/test.csv')

train.head()

The `fault_type` column is what we want to predict. The `id` column identifies each row for Kaggle, but it is not a measurement and should not be used as a model feature.

In [ ]:
feature_columns = [column for column in test.columns if column != 'id']
X_train = train[feature_columns]
y_train = train['fault_type']
X_test = test[feature_columns]

model = KNeighborsClassifier(n_neighbors=1)
model.fit(X_train, y_train)
predictions = model.predict(X_test)

predictions[:10]

In [ ]:
submission = pd.DataFrame({
    'id': test['id'],
    'fault_type': predictions,
})
submission.to_csv('submission.csv', index=False)
submission.head()

`submission.csv` is now ready to download from the file browser and upload to Kaggle. After making a first submission, try changing the number of neighbors, engineering features, or using a different classifier.